# GetDataset

In [1]:
#Install Libraries
!pip install -q transformers datasets evaluate torchaudio accelerate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 69.3 MB/s eta 0:00:00


In [2]:
from datasets import load_from_disk, Audio
from transformers import AutoProcessor, AutoModelForCTC, TrainingArguments, Trainer
import evaluate
import numpy as np
import torch

In [3]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"nalaprogroup","key":"6e69ae1b1b69e1a4d23d6edf0680d84f"}'}

In [4]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [5]:
!kaggle datasets download -d nalaprogroup/data-nalapro-project03

Dataset URL: https://www.kaggle.com/datasets/nalaprogroup/data-nalapro-project03
License(s): CC0-1.0
100% 354M/354M [00:02<00:00, 139MB/s]



In [6]:
import zipfile

zip_path = "data-nalapro-project03.zip"
extract_path = "/content/sample_data/data-nalapro-project03"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted to:", extract_path)

Extracted to: /content/sample_data/data-nalapro-project03


In [7]:
print("Loading dataset...")
dataset = load_from_disk(extract_path)
dataset

Loading dataset...


DatasetDict({
    train: Dataset({
        features: ['audio', 'intent_class', 'transcription'],
        num_rows: 1800
    })
    validation: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 56
    })
    test: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 57
    })
})

In [8]:
# Variable kan dataset
train_dataset = dataset["train"]
dev_dataset = dataset["validation"]
test_dataset = dataset["test"]

In [9]:
from datasets import DatasetDict


# =========================
# FINAL DATASET
# =========================
final_dataset = DatasetDict({
    "train": train_dataset,
    "validation": dev_dataset,
    "test": test_dataset
})

print(final_dataset)

DatasetDict({
    train: Dataset({
        features: ['audio', 'intent_class', 'transcription'],
        num_rows: 1800
    })
    validation: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 56
    })
    test: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 57
    })
})


In [10]:
print("Train:", len(final_dataset["train"]))
print("Dev:", len(final_dataset["validation"]))
print("Test:", len(final_dataset["test"]))

Train: 1800
Dev: 56
Test: 57


#Prerocessing
Konsep Dasar Whisper :
    Audio → Log-Mel Spectrogram → Encoder → Decoder → Text


In [11]:
# 1. Preprocessing (ASR Version)
from transformers import WhisperProcessor

# Processor menggabungkan Feature Extractor (audio) dan Tokenizer (teks)
processor = WhisperProcessor.from_pretrained("openai/whisper-base")

def prepare_dataset_asr(batch):
    # 1. Proses Audio (Input)
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=16000
    ).input_features[0]

    # 2. Proses Teks (Label/Target)
    # Tokenisasi teks transkripsi menjadi ID label
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch

# Jalankan mapping
encoded_dataset = final_dataset.map(
    prepare_dataset_asr,
    remove_columns=final_dataset["train"].column_names,
    num_proc=1
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/56 [00:00<?, ? examples/s]

Map:   0%|          | 0/57 [00:00<?, ? examples/s]

In [12]:
# Data Collator
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Padding input audio
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Padding label teks
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Ganti padding token ID dengan -100 agar tidak dihitung di loss function
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [13]:
# Evaluasi Menggunakan WER
import evaluate

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Ganti -100 agar bisa di-decode
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

In [14]:
#Load Dataset
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
# Training Arguments & Model
from transformers import WhisperForConditionalGeneration, TrainingArguments, Trainer, GenerationConfig
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base")

# Buat konfigurasi generasi yang bersih
generation_config = GenerationConfig.from_pretrained("openai/whisper-base")
generation_config.suppress_tokens = []
generation_config.forced_decoder_ids = None

# Masukkan kembali ke model
model.generation_config = generation_config

# Konfigurasi model
#model.config.forced_decoder_ids = None
# model.config.suppress_tokens = []

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/sample_data/Model/whisper-asr-minds14-01-Augm-epoch10",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    weight_decay=0.01,
    num_train_epochs=10,
    max_steps=-1,
    warmup_steps=500,
    fp16=True,

    # --- PERBAIKAN DI SINI ---
    eval_strategy="epoch",       # Evaluasi setiap akhir epoch agar pasti muncul
    save_strategy="epoch",       # Simpan model setiap akhir epoch
    logging_steps=10,            # Munculkan Training Loss setiap 10 langkah
    logging_strategy="steps",

    predict_with_generate=True,
    generation_max_length=225,
    metric_for_best_model="eval_wer",
    greater_is_better=False,
    report_to="none",

    # Tambahkan ini untuk memastikan log muncul di notebook
    disable_tqdm=False,
    load_best_model_at_end=True,
    save_total_limit=2,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    # --- TAMBAHKAN DI SINI ---
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2,
    early_stopping_threshold=0.1)]
)

# Cek apakah fungsi evaluasi berjalan sebelum training berat dimulai
print(trainer.evaluate())

trainer.train()

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

{'eval_loss': 5.207865238189697, 'eval_model_preparation_time': 0.0031, 'eval_wer': 41.838134430727024, 'eval_runtime': 12.99, 'eval_samples_per_second': 4.311, 'eval_steps_per_second': 0.539}


Epoch,Training Loss,Validation Loss,Model Preparation Time,Wer
1,2.839118,0.849575,0.003100,33.470508
2,0.868547,0.394420,0.003100,28.532236
3,0.360875,0.399283,0.003100,27.709191
4,0.060967,0.453796,0.003100,28.669410
5,0.020770,0.512922,0.003100,27.983539


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=565, training_loss=1.7360057378237226, metrics={'train_runtime': 1681.8016, 'train_samples_per_second': 10.703, 'train_steps_per_second': 0.672, 'total_flos': 5.8373996544e+17, 'train_loss': 1.7360057378237226, 'epoch': 5.0})

In [19]:
print(trainer.state.best_model_checkpoint)

/content/sample_data/Model/whisper-asr-minds14-01-Augm-epoch10/checkpoint-339


In [20]:
print(trainer.state.log_history)

[{'loss': 10.675906372070312, 'grad_norm': 157.322509765625, 'learning_rate': 1.8e-07, 'epoch': 0.08888888888888889, 'step': 10}, {'loss': 10.026061248779296, 'grad_norm': 127.75928497314453, 'learning_rate': 3.8e-07, 'epoch': 0.17777777777777778, 'step': 20}, {'loss': 9.901214599609375, 'grad_norm': 134.05279541015625, 'learning_rate': 5.800000000000001e-07, 'epoch': 0.26666666666666666, 'step': 30}, {'loss': 9.422113037109375, 'grad_norm': 111.28877258300781, 'learning_rate': 7.8e-07, 'epoch': 0.35555555555555557, 'step': 40}, {'loss': 7.870235443115234, 'grad_norm': 97.19461822509766, 'learning_rate': 9.800000000000001e-07, 'epoch': 0.4444444444444444, 'step': 50}, {'loss': 6.885236358642578, 'grad_norm': 87.45980834960938, 'learning_rate': 1.1800000000000001e-06, 'epoch': 0.5333333333333333, 'step': 60}, {'loss': 6.242145156860351, 'grad_norm': 76.74799346923828, 'learning_rate': 1.3800000000000001e-06, 'epoch': 0.6222222222222222, 'step': 70}, {'loss': 5.424560928344727, 'grad_nor

In [21]:
# Simpan hasil akhir model dan processor agar bisa dipakai lagi nanti
trainer.save_model("/content/drive/MyDrive/Colab Notebooks/Model/whisper-Augm-epoch10-EarlyStoper-FINAL")
processor.save_pretrained("/content/drive/MyDrive/Colab Notebooks/Model/whisper-Augm-epoch10-EarlyStoper-FINAL")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['/content/drive/MyDrive/Colab Notebooks/Model/whisper-Augm-epoch10-EarlyStoper-FINAL/processor_config.json']

In [ ]:
# TESTING INFERENCE
# 1. Cara Instan (Hugging Face Pipeline)
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from IPython.display import Audio, display

model_path = "/content/drive/MyDrive/Colab Notebooks/Model/whisper-Augm-epoch10-EarlyStoper-FINAL"

# 1. Cek ketersediaan perangkat (Device Agnostic)
# Jika GPU (CUDA) tersedia, gunakan "cuda". Jika tidak, gunakan "cpu".
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Menggunakan perangkat: {device}")

model = WhisperForConditionalGeneration.from_pretrained(model_path).to(device)
processor = WhisperProcessor.from_pretrained(model_path)

# 2. Ambil sampel (misalnya indeks ke-0 dari data test)
sample = final_dataset["test"][1]
audio_array = sample["audio"]["array"]
sampling_rate = sample["audio"]["sampling_rate"]

# 3. Tampilkan Pemutar Audio
print("Memutar Audio Asli:")
display(Audio(audio_array, rate=sampling_rate))

# 4. Proses Prediksi (Manual Inference)
input_features = processor(audio_array, sampling_rate=sampling_rate, return_tensors="pt").input_features.to(device)

with torch.no_grad():
    predicted_ids = model.generate(input_features)

# 5. Decode dan Tampilkan Hasil Teks
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

print(f"\n{'='*30}")
print(f"Hasil Prediksi : {transcription}")
print(f"Teks Sebenarnya: {sample['transcription']}")
print(f"{'='*30}")

Menggunakan perangkat: cpu


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Memutar Audio Asli:


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, 


Hasil Prediksi :  how do I set up a joint account
Teks Sebenarnya: how do I set up a joint account


In [ ]:
# 2. Cara Evaluasi Massal (Menguji Seluruh Test Set)
# Jalankan prediksi pada seluruh encoded_test
test_predictions = trainer.predict(encoded_dataset["test"])

# Ambil metriknya (termasuk WER)
print("Metrik pada Test Set:")
print(test_predictions.metrics)

In [ ]:
# Cara Manual (Input Audio Milik Sendiri)
import torch
import librosa
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from IPython.display import Audio, display

# Jika GPU (CUDA) tersedia, gunakan "cuda". Jika tidak, gunakan "cpu".
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Menggunakan perangkat: {device}")

# 1. Path ke model dan file audio Anda
model_path = "/content/drive/MyDrive/Colab Notebooks/Model/whisper-Augm-epoch10-EarlyStoper-FINAL"
path_audio_anda = "/content/sample_data/test_02.wav"  # Ganti dengan nama file yang Anda upload

# 2. Load Model & Processor
model = WhisperForConditionalGeneration.from_pretrained(model_path).to(device)
processor = WhisperProcessor.from_pretrained(model_path)

# 3. Load dan Resample Audio ke 16kHz
# librosa.load otomatis mengubah audio ke mono dan sampling rate yang kita tentukan
audio_input, sr = librosa.load(path_audio_anda, sr=16000)

# 4. Tampilkan Pemutar Audio untuk verifikasi
print("Mendengarkan file yang diuji:")
display(Audio(audio_input, rate=sr))

# 5. Preprocessing
input_features = processor(audio_input, sampling_rate=sr, return_tensors="pt").input_features.to(device)

# 6. Prediksi (Inference)
with torch.no_grad():
    predicted_ids = model.generate(input_features)

# 7. Decode Hasil
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

print(f"\n{'='*40}")
print(f"HASIL TRANSKRIPSI: {transcription}")
print(f"{'='*40}")

Menggunakan perangkat: cpu


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Mendengarkan file yang diuji:



HASIL TRANSKRIPSI:  it's so hard to see how it works


In [ ]:
# Cara Manual (Input Audio Milik Sendiri) - dengan bahasa indonesia
import torch
import librosa
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from IPython.display import Audio, display

# Jika GPU (CUDA) tersedia, gunakan "cuda". Jika tidak, gunakan "cpu".
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Menggunakan perangkat: {device}")

# 1. Path ke model dan file audio Anda
model_path = "/content/drive/MyDrive/Colab Notebooks/Model/whisper-Augm-epoch10-EarlyStoper-FINAL"
path_audio_anda = "/content/sample_data/test_02.wav"  # Ganti dengan nama file yang Anda upload

# 2. Load Model & Processor
model = WhisperForConditionalGeneration.from_pretrained(model_path).to(device)
processor = WhisperProcessor.from_pretrained(model_path)

# 3. Load dan Resample Audio ke 16kHz
# librosa.load otomatis mengubah audio ke mono dan sampling rate yang kita tentukan
audio_input, sr = librosa.load(path_audio_anda, sr=16000)

# 4. Tampilkan Pemutar Audio untuk verifikasi
print("Mendengarkan file yang diuji:")
display(Audio(audio_input, rate=sr))

# 5. Preprocessing
input_features = processor(audio_input, sampling_rate=sr, return_tensors="pt").input_features.to(device)

# 6. Prediksi (Inference) - Modifikasi untuk 2 Tugas
with torch.no_grad():
    # Jalankan untuk Transkripsi (Indonesia ke Indonesia)
    forced_ids_transcribe = processor.get_decoder_prompt_ids(language="indonesian", task="transcribe")
    predicted_ids_transcribe = model.generate(input_features, forced_decoder_ids=forced_ids_transcribe)

    # Jalankan untuk Translasi (Indonesia ke Inggris)
    forced_ids_translate = processor.get_decoder_prompt_ids(language="indonesian", task="translate")
    predicted_ids_translate = model.generate(input_features, forced_decoder_ids=forced_ids_translate)

    # Jalankan untuk Translasi (Default)
    predicted_ids = model.generate(input_features)

# 7. Decode Masing-masing Hasil
transcription_indo = processor.batch_decode(predicted_ids_transcribe, skip_special_tokens=True)[0]
transcription_eng = processor.batch_decode(predicted_ids_translate, skip_special_tokens=True)[0]
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

print("\n" + "="*40)
print(f"HASIL ASLI (INDO): {transcription_indo}")
print(f"HASIL TERJEMAHAN (ENG): {transcription_eng}")
print("="*40)
print(f"HASIL TERJEMAHAN (DEFAULT): {transcription}")

Menggunakan perangkat: cpu


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Mendengarkan file yang diuji:



HASIL ASLI (INDO):  aduh susah banget ya gimana ini jadi
HASIL TERJEMAHAN (ENG):  it's so hard to help how to do it
HASIL TERJEMAHAN (DEFAULT):  it's so hard to see how it works


In [ ]:
# Cara Manual (Input Audio Milik Sendiri) - otomatis
import torch
import librosa
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from IPython.display import Audio, display

# Jika GPU (CUDA) tersedia, gunakan "cuda". Jika tidak, gunakan "cpu".
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Menggunakan perangkat: {device}")

# 1. Path ke model dan file audio Anda
model_path = "/content/drive/MyDrive/Colab Notebooks/Model/whisper-Augm-epoch10-EarlyStoper-FINAL"
path_audio_anda = "/content/sample_data/test_01.wav"  # Ganti dengan nama file yang Anda upload

# 2. Load Model & Processor
model = WhisperForConditionalGeneration.from_pretrained(model_path).to(device)
processor = WhisperProcessor.from_pretrained(model_path)

# 3. Load dan Resample Audio ke 16kHz
# librosa.load otomatis mengubah audio ke mono dan sampling rate yang kita tentukan
audio_input, sr = librosa.load(path_audio_anda, sr=16000)

# 4. Tampilkan Pemutar Audio untuk verifikasi
print("Mendengarkan file yang diuji:")
display(Audio(audio_input, rate=sr))

# 5. Preprocessing
input_features = processor(audio_input, sampling_rate=sr, return_tensors="pt").input_features.to(device)


# 6. Prediksi (Inference) - Perbaikan Halusinasi Berulang
with torch.no_grad():
    # Ambil token ID untuk transcribe
    transcribe_token_id = processor.tokenizer.convert_tokens_to_ids("<|transcribe|>")

    predicted_ids = model.generate(
        input_features,
        max_new_tokens=225,
        # Mengunci tugas ke 'transcribe' agar TIDAK translate
        # Kita pasang di posisi 2 karena posisi 1 biasanya untuk bahasa
        forced_decoder_ids=[(2, transcribe_token_id)],

        # Mencegah pengulangan kata yang sama terus menerus
        no_repeat_ngram_size=3,

        # Mencegah model menghasilkan token kosong atau bising di awal
        begin_suppress_tokens=[processor.tokenizer.pad_token_id],

        # Memaksa model untuk lebih teliti (Beam Search) daripada tebakan instan
        num_beams=5,
        do_sample=False
    )

# 7. Decode Hasil
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

# Cek apakah dia berhasil deteksi bahasa
# Kita cek token di indeks ke-1 untuk melihat bahasa yang dipilih model
detected_lang_token = processor.tokenizer.decode(predicted_ids[0][1])
print(f"Bahasa yang dideteksi: {detected_lang_token}")

print("\n" + "="*40)
print(f"HASIL TRANSKRIPSI: {transcription}")
print("="*40)


Menggunakan perangkat: cpu


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Mendengarkan file yang diuji:


Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bahasa yang dideteksi: o

HASIL TRANSKRIPSI:  Helo! Морни! Я не т чекай myi myi кън бъм! can you help'em?
